In [1]:
import pandas as pd
import networkx as nx
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from EmailAnalytics import Email


In [2]:
def test_code(ea: Email):
    print("📌 Testing load_data...")
    df = ea.load_data()
    assert isinstance(df, pd.DataFrame)
    assert "text" in df.columns
    assert pd.api.types.is_datetime64_any_dtype(df["date"])
    assert len(df) == 18
    print("  ✔ load_data crea correctamente el DataFrame base.")

    print("\n📌 Testing build_interaction_graph...")
    graph = ea.build_interaction_graph(include_cc=True)
    assert isinstance(graph, nx.DiGraph)
    assert graph.has_edge("support.agent@ecorp.com", "alice.customer@northwind.com")
    assert graph.has_edge("support.agent@ecorp.com", "lead.support@ecorp.com")
    assert graph["support.agent@ecorp.com"]["lead.support@ecorp.com"]["weight"] == 1
    print("  ✔ build_interaction_graph devuelve un grafo dirigido válido.")

    print("\n📌 Testing analyze_sentiment...")
    df = ea.analyze_sentiment()
    for col in ["polarity", "subjectivity", "sentiment_label"]:
        assert col in df.columns
    assert df["polarity"].between(-1, 1).all()
    assert set(df["sentiment_label"]).issubset({"positive", "neutral", "negative"})
    print("  ✔ analyze_sentiment crea las columnas de sentimiento.")

    print("\n📌 Testing train_topic_model...")
    model, dictionary, corpus = ea.train_topic_model(num_topics=3, passes=10, random_state=42)
    assert isinstance(model, LdaModel)
    assert isinstance(dictionary, Dictionary)
    assert model.num_topics == 3
    assert len(corpus) == 18
    print("  ✔ train_topic_model entrena un modelo LDA válido.")

    print("\n📌 Testing assign_topics...")
    df = ea.assign_topics()
    assert "dominant_topic" in df.columns
    assert "topic_keywords" in df.columns
    assert len(df) == 18
    assert df["dominant_topic"].notna().all()
    print("  ✔ assign_topics anota un tema dominante por correo.")

    print("\n📌 Testing get_topic_report...")
    report = ea.get_topic_report(topn_words=5)
    assert isinstance(report, pd.DataFrame)
    for col in ["topic_id", "keywords", "num_emails", "mean_polarity"]:
        assert col in report.columns
    assert len(report) >= 1
    print("  ✔ get_topic_report devuelve un resumen estructurado por tema.")

    print("\n📌 Testing get_emails_by_sender...")
    subset = ea.get_emails_by_sender("hr@ecorp.com")
    assert isinstance(subset, pd.DataFrame)
    assert len(subset) == 2
    assert (subset["sender"] == "hr@ecorp.com").all()
    print("  ✔ get_emails_by_sender filtra correctamente por remitente.")

    print("\n📌 Testing get_emails_by_topic...")
    first_topic = int(ea.df["dominant_topic"].iloc[0])
    subset_topic = ea.get_emails_by_topic(first_topic)
    assert isinstance(subset_topic, pd.DataFrame)
    assert len(subset_topic) >= 1
    assert (subset_topic["dominant_topic"] == first_topic).all()
    print("  ✔ get_emails_by_topic filtra correctamente por tema.")

    print("\n📌 Testing graph_metrics...")
    metrics = ea.graph_metrics()
    assert isinstance(metrics, dict)
    for key in ["num_nodes", "num_edges", "density"]:
        assert key in metrics
    assert metrics["num_nodes"] > 0
    assert metrics["num_edges"] > 0
    print("  ✔ graph_metrics devuelve las métricas principales del grafo.")

    print("\n🎉 ALL TESTS HAVE BEEN EXECUTED SUCCESSFULLY.")


In [3]:
DATA_PATH = "correos_dataset.csv"
ea = Email(DATA_PATH)
test_code(ea)


📌 Testing load_data...
  ✔ load_data crea correctamente el DataFrame base.

📌 Testing build_interaction_graph...
  ✔ build_interaction_graph devuelve un grafo dirigido válido.

📌 Testing analyze_sentiment...
  ✔ analyze_sentiment crea las columnas de sentimiento.

📌 Testing train_topic_model...
  ✔ train_topic_model entrena un modelo LDA válido.

📌 Testing assign_topics...
  ✔ assign_topics anota un tema dominante por correo.

📌 Testing get_topic_report...
  ✔ get_topic_report devuelve un resumen estructurado por tema.

📌 Testing get_emails_by_sender...
  ✔ get_emails_by_sender filtra correctamente por remitente.

📌 Testing get_emails_by_topic...
  ✔ get_emails_by_topic filtra correctamente por tema.

📌 Testing graph_metrics...
  ✔ graph_metrics devuelve las métricas principales del grafo.

🎉 ALL TESTS HAVE BEEN EXECUTED SUCCESSFULLY.


In [4]:
display(ea.df)

,email_id,date,sender,recipients,cc,subject,body,text,polarity,subjectivity,sentiment_label,dominant_topic,topic_keywords
0,1,2026-01-03 09:15:00,alice.customer@northwind.com,support@ecorp.com,,Login error after password reset,"Hello team, I am unable to access the dashboar...","Login error after password reset Hello team, I...",-0.600000,0.350000,negative,1,"team, thank, error, login, twice"
1,2,2026-01-03 10:02:00,support.agent@ecorp.com,alice.customer@northwind.com,lead.support@ecorp.com,Re: Login error after password reset,"Hi Alice, thank you for reporting the issue. W...","Re: Login error after password reset Hi Alice,...",0.073016,0.728571,neutral,1,"team, thank, error, login, twice"
2,3,2026-01-04 08:41:00,bob.operations@northwind.com,support@ecorp.com,,Broken export feature in analytics panel,The export feature is broken again and the gen...,Broken export feature in analytics panel The e...,-0.475000,0.575000,negative,1,"team, thank, error, login, twice"
3,4,2026-01-04 12:10:00,support.agent@ecorp.com,bob.operations@northwind.com,,Re: Broken export feature in analytics panel,We deployed a patch for the export issue and i...,Re: Broken export feature in analytics panel W...,0.100000,0.333333,neutral,2,"invoice, looks, subscription, thank, incorrect"
4,5,2026-01-05 15:20:00,carla.it@northwind.com,support@ecorp.com,security@northwind.com,API timeout and connection failure,Our integration with your API is timing out an...,API timeout and connection failure Our integra...,-0.287500,0.316667,negative,0,"interview, next, role, would, analyst"
5,6,2026-01-05 17:05:00,lead.support@ecorp.com,carla.it@northwind.com,support.agent@ecorp.com,Re: API timeout and connection failure,We found the root cause of the timeout and the...,Re: API timeout and connection failure We foun...,0.136869,0.581818,positive,2,"invoice, looks, subscription, thank, incorrect"
6,7,2026-01-06 09:00:00,finance@bluewave.io,billing@ecorp.com,,Invoice 4831 looks incorrect,"Good morning, the invoice amount for January l...","Invoice 4831 looks incorrect Good morning, the...",0.133333,0.587500,positive,2,"invoice, looks, subscription, thank, incorrect"
7,8,2026-01-06 11:32:00,billing.agent@ecorp.com,finance@bluewave.io,,Re: Invoice 4831 looks incorrect,Thank you for the note. We checked the invoice...,Re: Invoice 4831 looks incorrect Thank you for...,0.000000,0.000000,neutral,2,"invoice, looks, subscription, thank, incorrect"
8,9,2026-01-07 08:14:00,accounts@greenrail.net,billing@ecorp.com,,Payment confirmation for February subscription,This email confirms that the payment for the F...,Payment confirmation for February subscription...,0.775000,0.975000,positive,2,"invoice, looks, subscription, thank, incorrect"
9,10,2026-01-07 13:47:00,billing.agent@ecorp.com,accounts@greenrail.net,finance.manager@ecorp.com,Re: Payment confirmation for February subscrip...,"Great, thank you for the confirmation. We are ...",Re: Payment confirmation for February subscrip...,0.566667,0.750000,positive,0,"interview, next, role, would, analyst"
